# Network Analysis: Centrality, Visualization, and Logistic Regression

```mermaid
flowchart TD
    A([Start]) --> B["Load graph: server URL and NetID to G"]
    B --> C["Compute degree, sampled unnormalized betweenness, PageRank, and peel"]
    C --> D["Assemble vertex_metrics indexed by vertex ID"]
    D --> E["Plot degree vs. peel; betweenness color and PageRank size"]
    D --> F["Compute six pairwise correlations"]
    D --> H["Label maximum-peel vertices: target 1, others 0"]
    H --> J["Fit standardized logistic regression"]
    J --> K["Return model and predictions; report training accuracy"]
    B --> M["Bonus: exact maximum clique IDs and labels"]
    E --> Z([End])
    F --> Z
    K --> Z
    M --> Z
```


### Dataset Description: Student Network Graph

The input is an undirected NetworkX graph `G`, fetched using `graph_client.get_graph(server_url, netid)`. 
Keep `graph_client.py` beside this notebook; fetching data requires access to the graph server.

**Set the server URL and your authorized NetID in the loading cell before running the notebook from top to bottom.**


- **Vertex ID**: the node key, retained as the `id` index in the vertex metrics and prediction tables.
- **`label`**: optional node attribute used as the display name; missing labels fall back to the vertex ID converted to a string.
- **Edges**: pairs of connected vertex IDs. The supplied client builds a simple undirected graph. Degree and betweenness are unweighted; PageRank uses edge weights if supplied. Peel and clique calculations remove self-loops on a copy.

Once the graph is loaded, you may save it to a local file with `save_graph(G, "graph.gpickle")` and reload it with `G = load_graph("graph.gpickle")`. **This is useful for large graphs or when the server is unavailable.**

## Tasks
1. Compute degree, betweenness centrality, PageRank, and peel value for every vertex.
2. Draw an interactive Plotly scatter plot with degree (x), peel (y), Viridis-colored betweenness, PageRank-sized points, hover details, logarithmic axes, pan dragging, scroll zoom, and an x-axis range slider.
3. Compute pairwise correlations.
4. Label maximum-peel vertices as 1, and all others as 0.
5. Fit a logistic regression using degree, betweenness, and PageRank.
6. **Bonus:** Find a maximum clique and print its vertex IDs and labels.

### Setup Code (Please run this first to set up the environment)
Run the setup cells first, then run the remaining cells in order. Keep `graph_client.py` beside this notebook.

In [1]:
from itertools import combinations

import networkx as nx
import pickle
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


from graph_client import get_graph

In [2]:
def save_graph(graph, filename):
    """
    Save a NetworkX graph to a local pickle file.
    IN: graph, nx.Graph, undirected graph with vertex IDs as node keys
        filename, str, destination path
    OUT: None; writes the graph to filename, replacing the file if it exists
    """
    with open(filename, 'wb') as f:
        pickle.dump(graph, f)

def load_graph(filename):
    """
    Load a previously saved NetworkX graph from a local pickle file.
    IN: filename, str, path to a graph saved by save_graph
    OUT: graph, nx.Graph, undirected graph with vertex IDs as node keys
    """
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [3]:
if __name__ == "__main__":
    G = get_graph("http://107.191.39.230:10439", netid="hz333")
    print(f"Vertices: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    print("Example vertex attributes:", next(iter(G.nodes(data=True)), None))
    print("Example edge attributes:", next(iter(G.edges(data=True)), None))

    # save_graph(G, "graph.gpickle")
    # G = load_graph("graph.gpickle")
    # print(f"Loaded graph: Vertices: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

Vertices: 1012, Edges: 112671
Example vertex attributes: (0, {'label': 'a cheap and fast cake'})
Example edge attributes: (0, 942, {})


### Task 1: Compute Vertex Metrics

#### Subtask 1.1: Compute Degree


Degree is the number of incident edge ends at a vertex; a self-loop contributes two to the degree.

In [4]:
def compute_degree(G):
    """
    Compute the degree of every vertex.
    IN: G, nx.Graph, undirected graph with vertex IDs as node keys
    OUT: degrees, dict[vertex ID, int], degree per vertex; a self-loop contributes two
    """
    return dict(G.degree())

#### Subtask 1.2: Compute Betweenness Centrality


Betweenness centrality sums, over pairs of other vertices, the fraction of shortest paths that pass through a vertex. Shortest paths are unweighted here.

This implementation estimates unnormalized betweenness using `k=128` sampled source vertices and `seed=42` for reproducibility. The sampled contributions are rescaled to estimate the full-graph scores.

Reference: [NetworkX documentation](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.centrality.betweenness_centrality.html#networkx.algorithms.centrality.betweenness_centrality)

In [5]:
def compute_betweenness(G, k=128, seed=42):
    """
    Compute betweenness centrality using unweighted shortest paths.
    IN: G, nx.Graph, undirected graph
        k, int or None, sampled source count (default 128); use None for exact scores;
        integer k must be between 1 and the number of vertices
        seed, int, random seed for reproducibility (default 42)
    OUT: betweenness, dict[vertex ID, float], estimated unnormalized betweenness score per vertex (exact if k=None)
    """
    return nx.betweenness_centrality(G, k=k, normalized=False, seed=seed)

#### Subtask 1.3: Compute PageRank


PageRank measures vertex importance through a random walk on the graph. On this undirected graph, each edge can be traversed in either direction. With damping factor `alpha=0.85`, the walk follows an edge with probability 0.85 and teleports with probability 0.15. Other settings use NetworkX defaults.

Reference: [NetworkX documentation](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.link_analysis.pagerank_alg.pagerank.html#networkx.algorithms.link_analysis.pagerank_alg.pagerank)


In [6]:
def compute_pagerank(G):
    """
    Compute PageRank with damping factor alpha=0.85.
    IN: G, nx.Graph, undirected graph; a weight attribute is used if present
    OUT: pagerank, dict[vertex ID, float], score per vertex
    """
    return nx.pagerank(G, alpha=0.85)

#### Subtask 1.4: Compute Peel Value (Core Number)


For each `k`, repeatedly remove vertices whose remaining degree is less than `k` to obtain the `k`-core. A vertex's peel value (core number) is the largest `k` for which it remains in a `k`-core. 

Reference: [NetworkX documentation](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.core.core_number.html#networkx.algorithms.core.core_number)

In [7]:
def compute_peel(G):
    """
    Compute each vertex's core number (peel value).
    IN: G, nx.Graph, graph converted to a simple undirected graph without self-loops
    OUT: peel, dict[vertex ID, int], largest k for which each vertex belongs to a k-core
    """
    H = nx.Graph(G)
    H.remove_edges_from(nx.selfloop_edges(H))
    return nx.core_number(H)

#### Subtask 1.5: Assemble Vertex Metrics

The output `vertex_metrics` has one row per vertex and index `id`. Its columns are `name` (display label), `degree`, `betweenness` (unnormalized, sampled by default), `pagerank`, and `peel` (peel value). The default betweenness calculation requires at least 128 vertices.


In [8]:
def build_vertex_metrics(G):
    """
    Assemble one row per vertex without changing G.
    IN: G, nx.Graph, undirected graph with optional node attribute label
    OUT: df, pd.DataFrame of shape (n_vertices, 5), index named id;
         columns name, degree, betweenness, pagerank, peel
    """

    nodes = list(G.nodes())
    degree = compute_degree(G)
    betweenness = compute_betweenness(G)
    pagerank = compute_pagerank(G)
    peel = compute_peel(G)

    df = pd.DataFrame({
        "name": [G.nodes[v].get("label", str(v)) for v in nodes],
        "degree": [degree[v] for v in nodes],
        "betweenness": [betweenness[v] for v in nodes],
        "pagerank": [pagerank[v] for v in nodes],
        "peel": [peel[v] for v in nodes],
    }, index=pd.Index(nodes, name="id"))
    
    return df

In [9]:
if __name__ == "__main__":
    vertex_metrics = build_vertex_metrics(G)
    print(vertex_metrics.head())

                      name  degree  betweenness  pagerank  peel
id                                                             
0    a cheap and fast cake     276   310.039729  0.001154   228
942                  sweet     438  2108.009907  0.001894   228
933                 strong     438  2108.009907  0.001894   228
450              alcoholic     436  2098.067320  0.001886   228
630               ethereal     438  2108.009907  0.001894   228


### Task 2: Interactive Scatter Plot
Plotly scatter plot with 
- degree (x), 
- peel (y), 
- Viridis-colored betweenness, 
- PageRank-sized points, 
- hover details, 
- logarithmic axes, 
- pan dragging, 
- scroll zoom, and 
- an x-axis range slider.

Reference: [Plotly documentation](https://plotly.com/python/)

In [10]:
def plot_vertex_metrics(df):
    """
    Display and return the interactive degree-versus-peel scatter plot.
    IN: df, pd.DataFrame, index named id and columns name, degree,
        betweenness, pagerank, peel; numeric metrics must be finite,
        pagerank must be nonnegative with at least one positive value in plotted rows
    OUT: fig, plotly.graph_objects.Figure, also displayed with scroll zoom;
         only rows with positive degree and peel are plotted
    RAISES: ValueError if no rows satisfy the logarithmic-axis requirements
    """

    plot_df = df[(df["degree"] > 0) & (df["peel"] > 0)].reset_index()
    if plot_df.empty or not (plot_df["pagerank"] > 0).any():
        raise ValueError("No rows with positive degree, peel and PageRank to plot on log axes.")

    fig = px.scatter(
        plot_df, x="degree", y="peel",
        color="betweenness", color_continuous_scale="Viridis",
        size="pagerank",
        hover_name="name",
        hover_data=["id", "degree", "peel", "betweenness", "pagerank"],
        log_x=True, log_y=True,
        title="Degree vs. Peel (color = betweenness, size = PageRank)",
    )
    fig.update_layout(dragmode="pan")
    fig.update_xaxes(rangeslider_visible=True)
    fig.show(config={"scrollZoom": True})
    
    return fig

In [21]:
if __name__ == "__main__":
    fig = plot_vertex_metrics(vertex_metrics)

### Task 3: Compute Pairwise Correlations
The example prints Pearson correlation coefficients for all six distinct pairs of degree, betweenness centrality, PageRank, and peel value (`peel`). A coefficient is NaN when the correlation is undefined, such as when a metric is constant.

In [12]:
def compute_correlation(df, metric_a, metric_b, method="pearson"):
    """
    Compute correlation between two metric columns.
    IN: df, pd.DataFrame, containing the requested numeric columns
        metric_a, str, first column name
        metric_b, str, second column name
        method, str, correlation method: pearson (default), spearman, or kendall
    OUT: correlation, float, coefficient in [-1, 1], or NaN if undefined
         (for example, a constant column or insufficient paired observations)
    """

    return float(df[metric_a].corr(df[metric_b], method=method))

In [13]:
if __name__ == "__main__":
    metric_columns = ["degree", "betweenness", "pagerank", "peel"]
    for col_a, col_b in combinations(metric_columns, 2):
        corr = compute_correlation(vertex_metrics, col_a, col_b)
        print(f"Correlation between {col_a} and {col_b}: {corr:.4f}")


Correlation between degree and betweenness: 0.7246
Correlation between degree and pagerank: 0.9901
Correlation between degree and peel: 0.9269
Correlation between betweenness and pagerank: 0.8031
Correlation between betweenness and peel: 0.5449
Correlation between pagerank and peel: 0.8993


### Task 4: Label Maximum-Peel Vertices

`label_max_peel` returns a copy named `labeled_metrics` with a binary `target` column: 1 for vertices whose peel equals the graph's maximum peel, and 0 for all other vertices. If all peel values are equal, every target is 1 and logistic regression cannot be fitted.


In [14]:
def label_max_peel(df):
    """
    Label vertices whose peel value equals the graph's maximum peel value.
    IN: df, pd.DataFrame, vertex metrics containing peel
    OUT: result, pd.DataFrame, copy of df with integer target column;
         1 indicates maximum peel, 0 indicates every other vertex
    """
    
    result = df.copy()
    result["target"] = (result["peel"] == result["peel"].max()).astype(int)
    return result

In [15]:
if __name__ == "__main__":
    labeled_metrics = label_max_peel(vertex_metrics)
    print("Number of vertices with max peel:", labeled_metrics["target"].sum())

Number of vertices with max peel: 483


### Task 5: Logistic Regression
Features: degree, betweenness, PageRank. Target: whether peel value equals the maximum. Standardization helps with features on very different scales. The model is fitted and evaluated on the same graph, so the reported training accuracy is **not** an estimate of out-of-sample performance. If only one class is present, logistic regression cannot be fitted.

In [16]:
def fit_logistic_regression(df):
    """
    Fit standardized logistic regression and predict on the same vertices.
    IN: df, pd.DataFrame, finite numeric degree, betweenness, pagerank columns,
        name column, and binary target column containing both 0 and 1
    OUT: model, sklearn.pipeline.Pipeline, fitted StandardScaler and LogisticRegression
         predictions, pd.DataFrame of shape (n_vertices, 4), same index as df;
         columns name, target, predicted (0 or 1), probability_1 (in [0, 1])
    """
    features = ["degree", "betweenness", "pagerank"]
    X = df[features]
    y = df["target"]

    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(X, y)

    predictions = df[["name", "target"]].copy()
    predictions["predicted"] = model.predict(X)
    predictions["probability_1"] = model.predict_proba(X)[:, 1]
    
    return model, predictions

In [17]:
if __name__ == "__main__":
    logistic_model, predictions = fit_logistic_regression(labeled_metrics)
    print("Training accuracy:", (predictions["target"] == predictions["predicted"]).mean())
    print(predictions.head())

Training accuracy: 0.9011857707509882
                      name  target  predicted  probability_1
id                                                          
0    a cheap and fast cake       1          1       0.753411
942                  sweet       1          1       0.992820
933                 strong       1          1       0.992820
450              alcoholic       1          1       0.992386
630               ethereal       1          1       0.992820


### Task 6 (Bonus): Find a Maximum Clique

In [18]:
def find_maximum_clique(G):
    """
    Find one largest clique.
    IN: G, nx.Graph, graph converted to a simple undirected graph without self-loops;
        optional node attribute label supplies display names
    OUT: (vertex_ids, vertex_labels), tuple of two aligned lists;
         missing labels fall back to str(vertex ID); empty graph returns ([], [])
         If multiple maximum cliques exist, returns one of them
    """

    H = nx.Graph(G)
    H.remove_edges_from(nx.selfloop_edges(H))
    if H.number_of_nodes() == 0:
        return [], []
    best = max(nx.find_cliques(H), key=len)

    return best, [G.nodes[v].get("label", str(v)) for v in best]

In [19]:
if __name__ == "__main__":
    clique_ids, clique_labels = find_maximum_clique(G)
    print("Maximum clique size:", len(clique_ids))
    print("Vertex IDs:", clique_ids)
    print("Vertex labels:", clique_labels)

Maximum clique size: 33
Vertex IDs: [250, 163, 370, 372, 259, 384, 434, 424, 129, 178, 416, 118, 179, 369, 343, 16, 659, 932, 709, 458, 971, 556, 461, 685, 560, 689, 945, 980, 820, 601, 698, 533, 695]
Vertex labels: ['mutton, popp with wine', 'fried ducks or teal', 'soft ginger cookies', 'soppor - clear broth soup', 'ortunga, frufwad', 'splattering-walling', 'wild strawberry jelly', 'to dye wool yellow on cotton and linen', 'dried green peas, to fopa', 'glazed bread', 'thin fluffy pancakes', 'creamed egg custard in a dish', 'godrån', 'soft filling in small waffle cups', 'shellac varnish, for finishing', 'aj wört', 'fruity', 'strawberry', 'humus', 'amber', 'tutti frutti', 'cherry', 'ammonia', 'guaiacol', 'cinnamic', 'hard yolk', 'tallow', 'vanillin', 'paint', 'crushedbug', 'herb', 'candy', 'heather']
